# Lab 4 — Build the Audience, Then Buy It
Name: Daryna Koval | Group: KM-42

## Setup

In [1]:
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 200)
DATA = kagglehub.dataset_download("shkarupylomaxim/adtech-lab4-tv-audiences/versions/2")
programs = pd.read_parquet(f"{DATA}/programs.parquet")
viewings = pd.read_parquet(f"{DATA}/viewings.parquet")
daily_weights = pd.read_parquet(f"{DATA}/daily_weights.parquet")
viewer_attributes = pd.read_parquet(f"{DATA}/viewer_attributes.parquet")
attributes = pd.read_parquet(f"{DATA}/attributes.parquet")
rate_card = pd.read_parquet(f"{DATA}/rate_card.parquet")
survey = pd.read_csv(f"{DATA}/purchase_survey.txt", sep="\t", dtype=str, keep_default_na=False)
survey_layout = pd.read_csv(f"{DATA}/purchase_survey_layout.txt", sep="\t", dtype=str)
acr_sessions = pd.read_parquet(f"{DATA}/acr_sessions.parquet")
hh_attributes = pd.read_parquet(f"{DATA}/hh_attributes.parquet")
reference_ratings = pd.read_parquet(f"{DATA}/reference_ratings.parquet")
universe = pd.read_parquet(f"{DATA}/universe.parquet")

c:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Task 1.0 — Clean the survey file

In [2]:
age = viewer_attributes[viewer_attributes["attr_id"] == 300].set_index("respondent_id")["value"].astype(float)
people_ids = viewer_attributes.loc[viewer_attributes["attr_id"].isin([101, 102]), "respondent_id"]
adults = people_ids[people_ids.map(age) >= 18]

s = survey.copy()
bad = []

junk = (s["RESPID"] == "RESPID") | s["RESPID"].str.startswith("TOTAL")
bad.append(["repeated header and footer lines", junk.sum(), "dropped"])
s = s[~junk]
print("Survey data rows:", len(s))

clean_id = s["RESPID"].str.strip().str.replace(".0", "", regex=False).astype(int)
bad.append(["RESPID with spaces, '.0' or a leading 0", (clean_id.astype(str) != s["RESPID"]).sum(), "fixed"])
s["RESPID"] = clean_id

clean_kw = s["KEYWORD"].str.strip().str.upper()
bad.append(["KEYWORD in lower case or with spaces", (clean_kw != s["KEYWORD"]).sum(), "fixed"])
s["KEYWORD"] = clean_kw

yes = ["1", "1.0", "y", "Y", "yes", "TRUE"]
bad.append(["VALUE written as y, yes, TRUE or 1.0", (s["VALUE"].isin(yes) & (s["VALUE"] != "1")).sum(), "read as yes"])
bad.append(["VALUE 0, no, N or blank", (~s["VALUE"].isin(yes)).sum(), "dropped"])
s = s[s["VALUE"].isin(yes)]

codes = survey_layout.loc[survey_layout["RECTYPE"] == "B", "KEYWORD"]
bad.append(["code not in the layout", (~s["KEYWORD"].isin(codes)).sum(), "dropped"])
s = s[s["KEYWORD"].isin(codes)]

bad.append(["id not an adult panel member", (~s["RESPID"].isin(adults)).sum(), "dropped"])
s = s[s["RESPID"].isin(adults)]

bad.append(["same person and code twice", s.duplicated(["RESPID", "KEYWORD"]).sum(), "dropped"])
s = s.drop_duplicates(["RESPID", "KEYWORD"])
print("Clean yes answers:", len(s))

pairs = s[s["KEYWORD"].isin(["X710101", "X710102", "X710103"])].copy()
pairs["n"] = pairs["KEYWORD"].str[-1].astype(int)

people = pd.DataFrame({"respondent_id": adults})
people["buyer"] = people["respondent_id"].isin(s["RESPID"])
people["pairs"] = people["respondent_id"].map(pairs.groupby("RESPID")["n"].max()).fillna(0).astype(int)
people["stride"] = people["respondent_id"].isin(s.loc[s["KEYWORD"] == "X710201", "RESPID"])
people["kestrel"] = people["respondent_id"].isin(s.loc[s["KEYWORD"] == "X710202", "RESPID"])
people["easywalk"] = people["respondent_id"].isin(s.loc[s["KEYWORD"] == "X710203", "RESPID"])

print("Adult panel members:", len(people))
print("Non-buyers:", (~people["buyer"]).sum())
print("People with two pair counts:", (pairs["RESPID"].value_counts() > 1).sum())
print(pd.DataFrame(bad, columns=["bad record", "rows", "what we did"]).to_string(index=False))

check = survey_layout[survey_layout["RECTYPE"] == "B"][["KEYWORD", "Text", "SAMPLE"]].copy()
our_count = []
for code in check["KEYWORD"]:
    if code == "X710100":
        our_count.append(people["buyer"].sum())
    elif code in ["X710101", "X710102", "X710103"]:
        our_count.append((people["pairs"] == int(code[-1])).sum())
    else:
        our_count.append(s.loc[s["KEYWORD"] == code, "RESPID"].nunique())
check["our count"] = our_count
print(check.to_string(index=False))

raw_kestrel = (survey["KEYWORD"] == "X710202").sum()
raw_buyers = survey.loc[survey["KEYWORD"] == "X710100", "RESPID"].nunique() / len(people)
print("Kestrel buyers, raw file vs clean:", raw_kestrel, people["kestrel"].sum())
print("Share of buyers, raw file vs clean:", round(raw_buyers, 3), round(people["buyer"].mean(), 3))

Survey data rows: 18346
Clean yes answers: 15510
Adult panel members: 8010
Non-buyers: 3231
People with two pair counts: 113
                             bad record  rows what we did
       repeated header and footer lines     3     dropped
RESPID with spaces, '.0' or a leading 0  1455       fixed
   KEYWORD in lower case or with spaces  1104       fixed
   VALUE written as y, yes, TRUE or 1.0  1194 read as yes
                VALUE 0, no, N or blank   742     dropped
                 code not in the layout   738     dropped
           id not an adult panel member   423     dropped
             same person and code twice   933     dropped
KEYWORD            Text SAMPLE  our count
X710100             Any   4779       4779
X710101          1 pair   1810       1810
X710102         2 pairs   1308       1308
X710103 3 or more pairs   1661       1661
X710201          STRIDE   1563       1563
X710202         KESTREL    354        354
X710203        EASYWALK   1050       1050
X710298     OTHER

**Answer:** The file did not keep the vendor's promises. It had header lines repeated inside the data and a TOTAL footer, ids and codes written in different ways (spaces, ".0", a leading zero, lower case), "yes" written in six forms, and answers that were not yes at all (0, no, blank). There were also codes that are not in the layout, ids of children, guests and unknown people, and 933 duplicate answers. I fixed the formats first and then dropped the bad rows; after that every count matches SAMPLE in the layout. With the raw file we would count 516 Kestrel rows instead of 354 buyers (46% too many), and only 54.8% of adults would be buyers instead of 59.7%.

## Task 1.1 — Four definitions

In [3]:
daily_weights["broadcast_date"] = pd.to_datetime(daily_weights["broadcast_date"])
programs["broadcast_date"] = pd.to_datetime(programs["broadcast_date"])

people["age"] = people["respondent_id"].map(age)
people["Adults 18+"] = True
people["Shoe buyers"] = people["buyer"]
people["Heavy buyers"] = people["pairs"] == 3
people["Kestrel buyers"] = people["kestrel"]
people["Adults 25-54"] = people["age"].between(25, 54)
audiences = ["Adults 18+", "Shoe buyers", "Heavy buyers", "Kestrel buyers", "Adults 25-54"]

daily = daily_weights.merge(people, on="respondent_id")
n_days = daily["broadcast_date"].nunique()

def size(aud):
    return daily.loc[daily[aud], "weight"].sum() / n_days

print("Adults 18+ (M):", round(size("Adults 18+") / 1e6, 1))
print("Shoe buyers (M):", round(size("Shoe buyers") / 1e6, 1))
print("Heavy buyers, 3+ pairs (M):", round(size("Heavy buyers") / 1e6, 1))
print("Kestrel buyers (M):", round(size("Kestrel buyers") / 1e6, 1))
print("Adults 25-54 (M):", round(size("Adults 25-54") / 1e6, 1))
print("Kestrel buyers in panel (n):", people["kestrel"].sum())

Adults 18+ (M): 252.6
Shoe buyers (M): 148.1
Heavy buyers, 3+ pairs (M): 49.6
Kestrel buyers (M): 10.0
Adults 25-54 (M): 123.7
Kestrel buyers in panel (n): 354


## Task 1.2 — Who are they?

In [4]:
men = viewer_attributes.loc[viewer_attributes["attr_id"] == 201, "respondent_id"]
daily["sex"] = "Women"
daily.loc[daily["respondent_id"].isin(men), "sex"] = "Men"
daily["age group"] = pd.cut(daily["age"], [17, 34, 54, 200], labels=["18-34", "35-54", "55+"]).astype(str)
daily["group"] = daily["sex"] + " " + daily["age group"]

share = pd.DataFrame()
for aud in audiences:
    d = daily[daily[aud]]
    share[aud] = d.groupby("group")["weight"].sum() / d["weight"].sum()
share = share.fillna(0)
index = share.div(share["Adults 18+"], axis=0) * 100

print("Share of each group in the audience:")
print(share.round(3))
print("Index vs adults 18+:")
print(index.round(0))

Share of each group in the audience:
             Adults 18+  Shoe buyers  Heavy buyers  Kestrel buyers  Adults 25-54
group                                                                           
Men 18-34         0.144        0.140         0.111           0.118         0.174
Men 35-54         0.160        0.158         0.138           0.189         0.326
Men 55+           0.185        0.186         0.189           0.126         0.000
Women 18-34       0.144        0.140         0.139           0.181         0.174
Women 35-54       0.160        0.178         0.226           0.265         0.326
Women 55+         0.208        0.197         0.196           0.121         0.000
Index vs adults 18+:
             Adults 18+  Shoe buyers  Heavy buyers  Kestrel buyers  Adults 25-54
group                                                                           
Men 18-34         100.0         97.0          77.0            82.0         121.0
Men 35-54         100.0         99.0          87.0 

**Answer:** I would give the planner **heavy buyers (3+ pairs)**. This group buys the most shoes, it is large enough to plan on (49.6 M people, 1,661 panelists), and it is clearly different from all adults: women 35-54 have index 142. Kestrel buyers are only 354 panelists, so their numbers are unreliable, and adults 25-54 does not fit the target: it leaves out the heavy buyers aged 55+, who are almost 40% of them. Athletic-shoe buyers look like all adults because about 60% of adults bought shoes; a group that large is almost the whole population, so all its indexes are close to 100. A fused purchase attribute cannot show if heavy buyers really prefer a program, because it is attached to people only through their age, sex and income, not through what they watch.

## Task 2.1 — Programs by audience

In [5]:
def minutes_watched(sessions, t):
    start = sessions["start_ts"].clip(lower=t["start_ts"])
    end = sessions["end_ts"].clip(upper=t["end_ts"])
    return ((end - start).dt.total_seconds() / 60).clip(lower=0)

def viewers(t, aud):
    s = pv_by_network[t["network_code"]]
    s = s[(s["start_ts"] < t["end_ts"]) & (s["end_ts"] > t["start_ts"])]
    if aud != "Persons 2+":
        s = s[s["respondent_id"].isin(people.loc[people[aud], "respondent_id"])]
    length = (t["end_ts"] - t["start_ts"]).total_seconds() / 60
    return (minutes_watched(s, t) * s["weight"]).sum() / length

def to_min(text):
    h, m = text.split(":")
    return int(h) * 60 + int(m) - 360

def price(t):
    rows = rate_card[rate_card["network_code"] == t["network_code"]]
    program_row = rows[rows["program_name"] == t["program_name"]]
    if len(program_row) > 0:
        return program_row["cost_30s_usd"].iloc[0]
    b = t["start_ts"] - pd.Timedelta(hours=6)
    minute = b.hour * 60 + b.minute
    for _, r in rows[rows["program_name"].isna()].iterrows():
        if b.dayofweek in list(r["weekdays"]) and to_min(r["start_time"]) <= minute < to_min(r["end_time"]):
            return r["cost_30s_usd"]

pv = viewings[viewings["resp_type"].isin(["R", "L"])]
pv_by_network = dict(list(pv.groupby("network_code")))
day_size = {}
for aud in audiences:
    day_size[aud] = daily[daily[aud]].groupby("broadcast_date")["weight"].sum()

eight = [["Sunday Night Football (NBC)", "NBC", "NFL Football"],
         ["Monday Night Football (ABC)", "ABC", "NFL Football"],
         ["Dancing with the Stars (ABC)", "ABC", "Dancing with the Stars"],
         ["NCIS (CBS)", "CBS", "NCIS"],
         ["Hannity (FNC)", "FNC", "Hannity"],
         ["Unsellable Houses (HGTV)", "HGTV", "Unsellable Houses"],
         ["Halloween Wars (FOOD)", "FOOD", "Halloween Wars"],
         ["Grotesquerie (FX)", "FX", "Grotesquerie"]]

rows = []
for label, net, name in eight:
    telecasts = programs[(programs["network_code"] == net) & (programs["program_name"] == name)
                         & (programs["daypart"] == "Prime")]
    for _, t in telecasts.iterrows():
        row = {"program": label, "cost": price(t)}
        for aud in ["Adults 18+", "Heavy buyers", "Kestrel buyers"]:
            row[aud] = viewers(t, aud)
            row[aud + " rating"] = row[aud] / day_size[aud][t["broadcast_date"]] * 100
        rows.append(row)
tel = pd.DataFrame(rows)
prog = tel.groupby("program", sort=False).mean()

table = pd.DataFrame()
table["adults rating"] = prog["Adults 18+ rating"]
table["target rating"] = prog["Heavy buyers rating"]
table["target viewers (000)"] = prog["Heavy buyers"] / 1000
table["composition"] = prog["Heavy buyers"] / prog["Adults 18+"]
table["index"] = prog["Heavy buyers rating"] / prog["Adults 18+ rating"] * 100
table["cost per :30"] = prog["cost"].round().astype(int)
table["adults CPM"] = prog["cost"] / prog["Adults 18+"] * 1000
table["target CPM"] = prog["cost"] / prog["Heavy buyers"] * 1000
print(table.round({"adults rating": 2, "target rating": 2, "target viewers (000)": 0, "composition": 3,
                   "index": 0, "adults CPM": 2, "target CPM": 2}))
print("Target universe (M):", round(size("Heavy buyers") / 1e6, 1))

                              adults rating  target rating  target viewers (000)  composition  index  cost per :30  adults CPM  target CPM
program                                                                                                                                   
Sunday Night Football (NBC)            5.82           5.79                2865.0        0.195   99.0        487200       33.12      170.05
Monday Night Football (ABC)            8.93           8.36                4149.0        0.184   94.0        723600       32.09      174.40
Dancing with the Stars (ABC)           3.25           3.14                1564.0        0.191   97.0        158600       19.33      101.39
NCIS (CBS)                             2.21           2.43                1206.0        0.216  110.0        178800       32.09      148.22
Hannity (FNC)                          1.09           0.84                 417.0        0.152   77.0         39900       14.50       95.71
Unsellable Houses (HGTV)   

## Task 2.2 — Index, volume or price?

In [6]:
print("Highest index (program):", table["index"].idxmax())
print("Most target viewers (program):", table["target viewers (000)"].idxmax())
print("Lowest target CPM (program):", table["target CPM"].idxmin())

Highest index (program): Unsellable Houses (HGTV)
Most target viewers (program): Monday Night Football (ABC)
Lowest target CPM (program): Unsellable Houses (HGTV)


**Answer:** No, they are different programs. Unsellable Houses (HGTV) has the highest index (127) and the lowest target CPM (40.66 USD), but only 317 thousand heavy buyers watch it in an average minute. Monday Night Football (ABC) has the most heavy buyers (4.1 M), but its index is below 100 and its target CPM is the highest (174.40 USD). This is the same as the shampoo example from Lecture 5: the channel with the highest share of buyers was not the one with the most buyers. I would tell the client that a high index means the money is used efficiently, but it does not give reach: with HGTV alone we would reach only a small part of the 49.6 M heavy buyers. A plan needs both efficient programs and big ones.

## Task 2.3 — Kestrel buyers

In [7]:
kestrel = pd.DataFrame()
kestrel["heavy-buyer index"] = table["index"]
kestrel["Kestrel index"] = prog["Kestrel buyers rating"] / prog["Adults 18+ rating"] * 100
print(kestrel.round(0))
print("Range (max - min):")
print((kestrel.max() - kestrel.min()).round(0))

                              heavy-buyer index  Kestrel index
program                                                       
Sunday Night Football (NBC)                99.0           99.0
Monday Night Football (ABC)                94.0           86.0
Dancing with the Stars (ABC)               97.0           83.0
NCIS (CBS)                                110.0           57.0
Hannity (FNC)                              77.0           43.0
Unsellable Houses (HGTV)                  127.0           77.0
Halloween Wars (FOOD)                     116.0           95.0
Grotesquerie (FX)                          96.0          130.0
Range (max - min):
heavy-buyer index    50.0
Kestrel index        87.0
dtype: float64


**Answer:** The Kestrel index goes from 43 to 130, while the heavy-buyer index stays between 77 and 127. The reason is the sample size: there are only 354 Kestrel buyers in the panel and 1,661 heavy buyers. On a small program only a few Kestrel buyers watch at the same time, and on some telecasts nobody does, so one person more or less changes the index a lot. I would trust the heavy-buyer index. To check the Kestrel index, I would count how many panelists are behind each number, compute its standard error, and compare the index in the first and second half of October: a real index should stay stable.

## Task 3.1 — Weights

In [8]:
acr = acr_sessions.copy()
acr["start_ts"] = pd.to_datetime(acr["start_utc"], utc=True).dt.tz_convert("America/New_York").dt.tz_localize(None)
acr["end_ts"] = pd.to_datetime(acr["end_utc"], utc=True).dt.tz_convert("America/New_York").dt.tz_localize(None)
acr["bday"] = (acr["start_ts"] - pd.Timedelta(hours=6)).dt.normalize()
acr["minutes"] = (acr["end_ts"] - acr["start_ts"]).dt.total_seconds() / 60
acr = acr[acr["minutes"] >= 1]

hh_per_ip = hh_attributes.groupby("hashed_ip")["household_id"].nunique()
tvs = acr_sessions[["tv_id", "hashed_ip"]].drop_duplicates("tv_id")
tvs = tvs[tvs["hashed_ip"].map(hh_per_ip) == 1].merge(hh_attributes, on="hashed_ip")
tvs["head_age"] = "Under 65"
tvs.loc[tvs["head_age_band"] == "65+", "head_age"] = "65+"
print("TVs matched to one household:", len(tvs))

intab = acr[["tv_id", "bday"]].drop_duplicates().merge(tvs[["tv_id", "head_age", "income_band"]], on="tv_id")
intab["tvs in cell"] = intab.groupby(["bday", "head_age", "income_band"])["tv_id"].transform("count")
intab = intab.merge(universe, on=["head_age", "income_band"])
intab["weight"] = intab["tv_households"] / intab["tvs in cell"]
weight_sum = intab.groupby("bday")["weight"].sum() / 1e6
print("Weights sum per day (M):", weight_sum.round(1).min(), "-", weight_sum.round(1).max())

live = acr[(acr["session_type"] == "live") & acr["tv_id"].isin(tvs["tv_id"])]
ref = reference_ratings.merge(programs, on="telecast_id")
rows = []
for _, t in ref.iterrows():
    s = live[live["network_code"] == t["network_code"]]
    watched = s.loc[minutes_watched(s, t) >= 1, "tv_id"]
    day = intab[intab["bday"] == t["broadcast_date"]]
    saw = day["tv_id"].isin(watched)
    rows.append({"telecast": t["program_name"] + " (" + t["network_code"] + ")",
                 "unweighted": saw.mean() * 100,
                 "weighted": day.loc[saw, "weight"].sum() / day["weight"].sum() * 100,
                 "reference": t["hh_rating"]})
compare = pd.DataFrame(rows).set_index("telecast")
print(compare.round(2))

gap_unweighted = (compare["reference"] - compare["unweighted"]).abs().mean()
gap_weighted = (compare["reference"] - compare["weighted"]).abs().mean()
print("Mean gap unweighted (points):", round(gap_unweighted, 2))
print("Mean gap weighted (points):", round(gap_weighted, 2))
print("Share of gap closed:", round(1 - gap_weighted / gap_unweighted, 3))

TVs matched to one household: 19607
Weights sum per day (M): 125.8 - 125.8
                                              unweighted  weighted  reference
telecast                                                                     
MLB Postseason (FOX)                               17.03     17.76      25.86
NFL Football (NBC)                                 22.62     23.75      33.14
NCIS (CBS)                                          5.50      5.80      10.56
Hannity (FNC)                                       1.04      1.35       2.30
Unsellable Houses (HGTV)                            2.12      2.08       3.69
Halloween Baking Championship (FOOD)                0.95      0.90       1.63
Grotesquerie (FX)                                   2.17      2.11       3.53
The Real Housewives of New York City (BRAVO)        0.71      0.64       1.23
Mean gap unweighted (points): 3.73
Mean gap weighted (points): 3.44
Share of gap closed: 0.075


## Task 3.2 — What weighting cannot fix
**Answer:** After weighting, the ratings of programs with older viewers went up (World Series on FOX, Sunday Night Football, NCIS, Hannity), and the young cable shows went slightly down. This happened because homes with a head aged 65+ are under-represented among the smart TVs, so they got bigger weights. But weighting closed only 7.5% of the gap, and all ratings are still 28-48% below the reference. Weighting only changes how much each home counts; it cannot add viewing that the data does not have. We see only one TV per home, only live viewing that the TV recognized, and we dropped the TVs without a matched household. Weighting also cannot fix the fact that people who opt in may watch differently from others of the same age and income (Lecture 4). Income barely changes the ratings because, within the same age group, people with different incomes watch these programs about the same.

## Task 4.1 — October predicts October

In [9]:
all_persons = daily_weights[daily_weights["resp_type"].isin(["R", "L"])]
day_size["Persons 2+"] = all_persons.groupby("broadcast_date")["weight"].sum()

def rating(t, aud):
    return viewers(t, aud) / day_size[aud][t["broadcast_date"]] * 100

big5 = ["CBS", "NBC", "ABC", "FOX", "FNC"]
prime = programs[programs["network_code"].isin(big5) & (programs["daypart"] == "Prime")
                 & (programs["broadcast_date"] <= "2024-10-28")].copy()
prime["P2+"] = [rating(t, "Persons 2+") for _, t in prime.iterrows()]
prime["heavy"] = [rating(t, "Heavy buyers") for _, t in prime.iterrows()]

history = prime[prime["broadcast_date"] <= "2024-10-21"]
forecast = history.groupby(["network_code", "program_name"])[["P2+", "heavy"]].mean()
week = prime[prime["broadcast_date"] >= "2024-10-22"]
week = week.merge(forecast, on=["network_code", "program_name"], suffixes=("", " forecast"))
print("Telecasts forecast (n):", len(week))

for aud in ["P2+", "heavy"]:
    actual = week[aud].where(week[aud] > 0)
    week[aud + " error"] = (actual - week[aud + " forecast"]).abs() / actual
print("MAPE, P2+ (%):", round(week["P2+ error"].mean() * 100, 1))
print("MAPE, heavy buyers (%):", round(week["heavy error"].mean() * 100, 1))

week["program"] = week["program_name"] + " (" + week["network_code"] + ")"
week["date"] = week["broadcast_date"].dt.strftime("%Y-%m-%d")
week = week.sort_values("P2+ error", ascending=False)
print("Biggest miss, P2+ (program, date):", week["program"].iloc[0], week["date"].iloc[0])

Telecasts forecast (n): 101
MAPE, P2+ (%): 35.8
MAPE, heavy buyers (%): 38.6
Biggest miss, P2+ (program, date): Krapopolis (FOX) 2024-10-27


## Task 4.2 — Why it missed

In [10]:
week["start"] = week["start_ts"].dt.strftime("%H:%M")
print(week[["program", "date", "start", "P2+ forecast", "P2+"]].head(5).round(2).to_string(index=False))

                   program       date start  P2+ forecast  P2+
          Krapopolis (FOX) 2024-10-27 21:30          1.06 0.22
Universal Basic Guys (FOX) 2024-10-27 20:30          0.68 0.26
        Fire Country (CBS) 2024-10-27 22:44          1.70 0.67
             Accused (FOX) 2024-10-22 21:00          0.95 0.37
  What Would You Do? (ABC) 2024-10-23 22:00          0.44 0.18


**Answer:** Most big misses come from schedule changes that an average cannot see. Krapopolis and Universal Basic Guys on FOX (October 27) had aired in other time slots before, and on October 20 Universal Basic Guys had a big NFL game before it; on October 27 there was no such lead-in. On CBS the football game ran late on October 27, so Fire Country was moved and split into short parts. Accused had a strong premiere and then lost viewers every week, and What Would You Do? was also falling, so the average stayed too high. The heavy-buyer forecast is worse (38.6% against 35.8%) because it is based on 1,661 panelists instead of all persons, so the numbers are noisier (Lecture 6). A better forecast would use the time slot, the day and the lead-in of each telecast, and the trend of a show instead of a simple average.

## Task 5.1 — The plan

In [11]:
plan = pd.DataFrame({"spots": {"Dancing with the Stars (ABC)": 2, "NCIS (CBS)": 1, "Hannity (FNC)": 5,
                               "Unsellable Houses (HGTV)": 9, "Halloween Wars (FOOD)": 6, "Grotesquerie (FX)": 12}})
plan["cost"] = plan["spots"] * table.loc[plan.index, "cost per :30"]
plan["target impressions (M)"] = plan["spots"] * table.loc[plan.index, "target viewers (000)"] / 1000
plan["adult impressions (M)"] = plan["target impressions (M)"] / table.loc[plan.index, "composition"]
print(plan.round(2))
print("Plan cost ($):", plan["cost"].sum())
print("Plan target impressions (M):", round(plan["target impressions (M)"].sum(), 1))
print("Plan target CPM ($):", round(plan["cost"].sum() / plan["target impressions (M)"].sum() / 1000, 2))
print("Plan composition:", round(plan["target impressions (M)"].sum() / plan["adult impressions (M)"].sum(), 3))

                              spots    cost  target impressions (M)  adult impressions (M)
Dancing with the Stars (ABC)      2  317200                    3.13                  16.41
NCIS (CBS)                        1  178800                    1.21                   5.57
Hannity (FNC)                     5  199500                    2.08                  13.76
Unsellable Houses (HGTV)          9  116100                    2.86                  11.48
Halloween Wars (FOOD)             6   58200                    0.97                   4.26
Grotesquerie (FX)                12   91104                    1.49                   7.92
Plan cost ($): 960904
Plan target impressions (M): 11.7
Plan target CPM ($): 81.87
Plan composition: 0.198


## Task 5.2 — Recommendation
**Answer:** I would spend 960,904 USD on 35 spots in six programs and get about 11.7 M heavy-buyer impressions at a target CPM of 81.87 USD. Most spots go to the cable shows with a high index and a low target CPM (Unsellable Houses, Halloween Wars, Grotesquerie), and a few go to bigger programs (Dancing with the Stars, NCIS, Hannity) to add volume. I did not buy football, because its target CPM is about 170 USD and one spot would take about half of the budget or more. The delivery is not certain: our forecast missed heavy-buyer ratings by 38.6% on average, so I would ask the networks for an audience guarantee with make-goods and keep the remaining 39,096 USD as a reserve. Only about 20% of the people who see the ads are heavy buyers (composition 0.198), so about 80% of the money reaches other people, which is normal for TV. After the week I would compare the delivered heavy-buyer ratings with the plan, check how many heavy buyers we reached and how often, and ask for make-goods where a program delivered less.

## Bonus

In [12]:
print("Kaggle username:", "not submitted")

Kaggle username: not submitted


**Answer:** I did not take part in the Kaggle competition.